# Prédiction des résultats de la Ligue 1 — Saison 2025-2026

## Objectif
Construire un classifieur qui prédit, pour chaque match de Ligue 1 de la saison 2025-2026, l'un des trois résultats suivants :
- **`1`** : victoire de l'équipe à domicile
- **`0`** : match nul
- **`-1`** : victoire de l'équipe à l'extérieur

## Plan du notebook
1. Chargement et exploration des données
2. Feature engineering — on construit des **variables explicatives** sans fuite de données :
   - ELO mis à jour match par match
   - Forme récente (fenêtre glissante de 5 matchs)
   - Confrontations directes (head-to-head)
   - Valeur marchande des effectifs
   - Caractéristiques statiques des clubs
3. Entraînement et comparaison de plusieurs modèles
4. Validation temporelle : on entraîne sur les saisons anciennes et on valide sur les plus récentes
5. Génération du fichier `predictions.csv`

## Note méthodologique importante
Toutes les features sont calculées **uniquement à partir des matchs antérieurs** au match en cours. C'est crucial pour éviter une **fuite de données** (data leakage) : si on utilisait des infos futures, le score en validation serait artificiellement bon et le modèle s'effondrerait sur les vraies prédictions 2025-2026.

## 1. Imports et configuration

In [ ]:
import re
from collections import defaultdict, deque
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, log_loss
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Dossier où se trouvent les CSV (à adapter si nécessaire)
DATA_DIR = Path('.')

pd.set_option('display.max_columns', 50)

## 2. Chargement des données

On dispose de plusieurs fichiers CSV. Pour ce projet on utilise principalement :
- `matchs_2013_2024.csv` : matchs passés avec leurs résultats (notre jeu d'entraînement)
- `match_2025.csv` : matchs à prédire (sans résultat)
- `clubs_fr.csv` : caractéristiques statiques des clubs de Ligue 1 (effectif, âge moyen, etc.)
- `player_valuation_before_season.csv` : valeurs marchandes des joueurs aux différentes dates

On **filtre** sur `competition_type == 'domestic_league'` pour ne garder que les matchs de championnat (pas la Coupe ni les compétitions européennes).

On **trie chronologiquement** : c'est indispensable pour calculer les features incrémentales (ELO, forme) sans fuite de données.

In [ ]:
matches_hist = pd.read_csv(DATA_DIR / 'matchs_2013_2024.csv')
matches_2025 = pd.read_csv(DATA_DIR / 'match_2025.csv')
clubs = pd.read_csv(DATA_DIR / 'clubs_fr.csv')
valuations = pd.read_csv(DATA_DIR / 'player_valuation_before_season.csv')

# On garde uniquement les matchs de championnat (Ligue 1)
matches_hist = matches_hist[matches_hist['competition_type'] == 'domestic_league'].copy()

# Conversion des dates au bon format pour pouvoir trier et calculer des deltas
matches_hist['date'] = pd.to_datetime(matches_hist['date'])
matches_2025['date'] = pd.to_datetime(matches_2025['date'])
valuations['date'] = pd.to_datetime(valuations['date'])

# Tri chronologique : crucial pour les calculs incrémentaux (ELO, forme) sans fuite
matches_hist = matches_hist.sort_values('date').reset_index(drop=True)
matches_2025 = matches_2025.sort_values('date').reset_index(drop=True)

print(f'Matchs historiques : {len(matches_hist)}')
print(f'Matchs à prédire   : {len(matches_2025)}')
matches_hist.head(3)

## 3. Exploration rapide

Avant de modéliser, on regarde la distribution des résultats. C'est la **baseline** à battre : si on prédit toujours la classe majoritaire, on a déjà un certain score gratuit.

In [ ]:
print('Distribution des résultats (1 = dom, 0 = nul, -1 = ext) :')
print(matches_hist['results'].value_counts(normalize=True).round(3))
print()
print('=> L\'avantage du domicile est marqué : ~44% de victoires à domicile.')
print('=> Si on prédit toujours "1", on a 44% de bonnes réponses : c\'est la baseline à battre.')

## 4. Feature engineering

C'est le cœur du projet. On va construire 5 familles de features.

### 4.1 ELO incrémental

Le **classement ELO** est une note attribuée à chaque équipe, inspirée du classement aux échecs. Le principe :
- Chaque équipe démarre à 1500.
- Après chaque match, on calcule la probabilité attendue de victoire en fonction de la différence d'ELO (+ un bonus pour l'équipe à domicile).
- L'ELO du gagnant monte, celui du perdant descend. L'amplitude dépend de l'écart par rapport à ce qui était attendu (battre un favori rapporte plus que battre un outsider).

C'est une variable **très informative** en sport car elle résume l'historique complet d'une équipe en un seul nombre, et elle s'adapte au fil du temps.

**Astuce anti-fuite** : on enregistre l'ELO **avant** la mise à jour. Ainsi le modèle ne connaît jamais le résultat du match qu'il cherche à prédire.

In [ ]:
def compute_elo_features(matches: pd.DataFrame, k: float = 20.0, home_adv: float = 80.0):
    """Calcule l'ELO de chaque équipe AVANT chaque match.

    Paramètres:
      k         : sensibilité de la mise à jour (plus élevé = plus réactif)
      home_adv  : bonus appliqué à l'équipe à domicile dans le calcul de la proba attendue
    """
    elo = defaultdict(lambda: 1500.0)  # toute équipe inconnue démarre à 1500
    home_elos, away_elos = [], []

    for row in matches.itertuples(index=False):
        h, a = row.home_club_id, row.away_club_id
        eh, ea = elo[h], elo[a]

        # On enregistre l'ELO AVANT le match (sinon fuite de données)
        home_elos.append(eh)
        away_elos.append(ea)

        # Proba attendue de victoire à domicile (formule ELO classique)
        exp_home = 1.0 / (1.0 + 10 ** (-(eh + home_adv - ea) / 400))

        # Score réel : 1 = victoire dom, 0.5 = nul, 0 = défaite dom
        if row.results == 1:
            score_home = 1.0
        elif row.results == 0:
            score_home = 0.5
        else:
            score_home = 0.0

        # Mise à jour : l'ELO bouge proportionnellement à l'écart entre réalité et attente
        elo[h] = eh + k * (score_home - exp_home)
        elo[a] = ea + k * ((1 - score_home) - (1 - exp_home))

    return pd.DataFrame({'home_elo': home_elos, 'away_elo': away_elos}), dict(elo)

elo_df, elo_state = compute_elo_features(matches_hist)
matches_hist = pd.concat([matches_hist.reset_index(drop=True), elo_df], axis=1)

# Pour les matchs 2025-2026 : on utilise l'ELO final calculé sur l'historique
matches_2025['home_elo'] = matches_2025['home_club_id'].map(lambda c: elo_state.get(c, 1500.0))
matches_2025['away_elo'] = matches_2025['away_club_id'].map(lambda c: elo_state.get(c, 1500.0))

# Top 5 équipes par ELO actuel
elo_ranking = pd.Series(elo_state).sort_values(ascending=False).head(10)
id_to_name = dict(zip(matches_hist['home_club_id'], matches_hist['home_club_name']))
id_to_name.update(dict(zip(matches_hist['away_club_id'], matches_hist['away_club_name'])))
print('Top 10 ELO en fin d\'historique :')
for cid, score in elo_ranking.items():
    print(f'  {id_to_name.get(cid, cid):40s} {score:.0f}')

### 4.2 Forme récente (rolling window)

Pour chaque équipe, on calcule sa **performance sur ses 5 derniers matchs** : points gagnés, buts marqués, buts encaissés. Idée : une équipe sur une bonne dynamique a tendance à continuer (et inversement).

**Implémentation** : on parcourt les matchs dans l'ordre chronologique en maintenant, pour chaque équipe, une `deque` (file à taille max) qui conserve uniquement les 5 derniers matchs. On lit la forme avant le match, puis on met à jour après.

In [ ]:
def compute_form_features(matches: pd.DataFrame, window: int = 5):
    """Calcule les features de forme récente pour chaque équipe avant chaque match."""
    history = defaultdict(lambda: deque(maxlen=window))
    rows = []

    for row in matches.itertuples(index=False):
        feats = {}
        for side, club in [('home', row.home_club_id), ('away', row.away_club_id)]:
            past = list(history[club])
            if past:
                feats[f'{side}_form_pts'] = np.mean([p['pts'] for p in past])
                feats[f'{side}_form_gf']  = np.mean([p['gf']  for p in past])  # buts marqués
                feats[f'{side}_form_ga']  = np.mean([p['ga']  for p in past])  # buts encaissés
                feats[f'{side}_form_gd']  = feats[f'{side}_form_gf'] - feats[f'{side}_form_ga']
                feats[f'{side}_form_n']   = len(past)
            else:
                # Première apparition de l'équipe : pas d'historique
                for k in ('pts', 'gf', 'ga', 'gd'):
                    feats[f'{side}_form_{k}'] = np.nan
                feats[f'{side}_form_n'] = 0
        rows.append(feats)

        # Mise à jour APRÈS avoir lu la forme (pas avant !)
        if row.results == 1:
            pts_h, pts_a = 3, 0
        elif row.results == 0:
            pts_h, pts_a = 1, 1
        else:
            pts_h, pts_a = 0, 3
        history[row.home_club_id].append({'pts': pts_h, 'gf': row.home_club_goals, 'ga': row.away_club_goals})
        history[row.away_club_id].append({'pts': pts_a, 'gf': row.away_club_goals, 'ga': row.home_club_goals})

    return pd.DataFrame(rows), history

form_df, form_state = compute_form_features(matches_hist, window=5)
matches_hist = pd.concat([matches_hist.reset_index(drop=True), form_df], axis=1)

# Pour les matchs 2025 : on prend la forme finale de chaque équipe
def attach_form_future(future, history, window=5):
    rows = []
    for row in future.itertuples(index=False):
        feats = {}
        for side, club in [('home', row.home_club_id), ('away', row.away_club_id)]:
            past = list(history.get(club, []))
            if past:
                feats[f'{side}_form_pts'] = np.mean([p['pts'] for p in past])
                feats[f'{side}_form_gf']  = np.mean([p['gf']  for p in past])
                feats[f'{side}_form_ga']  = np.mean([p['ga']  for p in past])
                feats[f'{side}_form_gd']  = feats[f'{side}_form_gf'] - feats[f'{side}_form_ga']
                feats[f'{side}_form_n']   = len(past)
            else:
                for k in ('pts', 'gf', 'ga', 'gd'):
                    feats[f'{side}_form_{k}'] = np.nan
                feats[f'{side}_form_n'] = 0
        rows.append(feats)
    return pd.DataFrame(rows)

matches_2025 = pd.concat([matches_2025.reset_index(drop=True), attach_form_future(matches_2025, form_state)], axis=1)
matches_hist[['date', 'home_club_name', 'away_club_name', 'home_form_pts', 'away_form_pts', 'home_elo', 'away_elo']].tail(5)

### 4.3 Head-to-head (confrontations directes)

Certains matchs ont une **histoire** : Marseille vs PSG, OL vs Saint-Étienne, etc. On calcule, pour chaque paire d'équipes, le bilan de leurs **5 dernières confrontations** (toutes éditions confondues, à domicile ou non).

**Astuce** : on indexe par paire triée `tuple(sorted([id1, id2]))` pour que le même historique soit consulté quel que soit qui reçoit.

In [ ]:
def compute_h2h_features(matches: pd.DataFrame, window: int = 5):
    history = defaultdict(lambda: deque(maxlen=window))
    rows = []

    for row in matches.itertuples(index=False):
        key = tuple(sorted([row.home_club_id, row.away_club_id]))
        past = list(history[key])
        if past:
            home_wins = sum(1 for p in past if p['winner'] == row.home_club_id)
            draws     = sum(1 for p in past if p['winner'] == 0)
            n = len(past)
            rows.append({'h2h_home_winrate': home_wins / n,
                         'h2h_draw_rate':    draws / n,
                         'h2h_n':            n})
        else:
            rows.append({'h2h_home_winrate': np.nan, 'h2h_draw_rate': np.nan, 'h2h_n': 0})

        # Mise à jour
        if   row.results ==  1: winner = row.home_club_id
        elif row.results == -1: winner = row.away_club_id
        else:                   winner = 0  # nul
        history[key].append({'winner': winner})

    return pd.DataFrame(rows), history

h2h_df, h2h_state = compute_h2h_features(matches_hist)
matches_hist = pd.concat([matches_hist.reset_index(drop=True), h2h_df], axis=1)

def attach_h2h_future(future, history):
    rows = []
    for row in future.itertuples(index=False):
        key = tuple(sorted([row.home_club_id, row.away_club_id]))
        past = list(history.get(key, []))
        if past:
            home_wins = sum(1 for p in past if p['winner'] == row.home_club_id)
            draws     = sum(1 for p in past if p['winner'] == 0)
            n = len(past)
            rows.append({'h2h_home_winrate': home_wins / n, 'h2h_draw_rate': draws / n, 'h2h_n': n})
        else:
            rows.append({'h2h_home_winrate': np.nan, 'h2h_draw_rate': np.nan, 'h2h_n': 0})
    return pd.DataFrame(rows)

matches_2025 = pd.concat([matches_2025.reset_index(drop=True), attach_h2h_future(matches_2025, h2h_state)], axis=1)

### 4.4 Valeur marchande des effectifs

Intuition : une équipe avec des joueurs chers est en moyenne plus performante. On calcule pour chaque (club, saison) :
- **`squad_value`** : somme des valeurs marchandes de tous les joueurs
- **`squad_top3_value`** : somme des 3 joueurs les plus chers (proxy de l'attaque vedette)
- **`squad_n_players`** : nombre de joueurs valorisés

**Convention de saison** : la saison `N` couvre août `N` → mai `N+1`. On rattache donc une valuation à sa saison via l'année calendaire.

In [ ]:
valuations['year'] = valuations['date'].dt.year

rows = []
for (club, year), grp in valuations.groupby(['current_club_id', 'year']):
    # Pour chaque joueur : sa dernière valeur connue dans l'année
    latest_per_player = grp.sort_values('date').groupby('player_id').tail(1)
    rows.append({
        'club_id': club,
        'season': year,
        'squad_value':      latest_per_player['market_value_in_eur'].sum(),
        'squad_top3_value': latest_per_player['market_value_in_eur'].nlargest(3).sum(),
        'squad_n_players':  len(latest_per_player),
    })
squad_vals = pd.DataFrame(rows)

def attach_squad_values(matches, squad_vals):
    out = matches.copy()
    if 'season' not in out.columns:
        # Pour les matchs futurs : déduire la saison (août–décembre => année, sinon année-1)
        m = out['date'].dt.month
        out['season'] = np.where(m >= 7, out['date'].dt.year, out['date'].dt.year - 1)

    # Jointure côté domicile
    out = out.merge(
        squad_vals.rename(columns={'club_id': 'home_club_id',
                                    'squad_value': 'home_squad_value',
                                    'squad_top3_value': 'home_squad_top3',
                                    'squad_n_players': 'home_squad_n'}),
        on=['home_club_id', 'season'], how='left',
    )
    # Jointure côté extérieur
    out = out.merge(
        squad_vals.rename(columns={'club_id': 'away_club_id',
                                    'squad_value': 'away_squad_value',
                                    'squad_top3_value': 'away_squad_top3',
                                    'squad_n_players': 'away_squad_n'}),
        on=['away_club_id', 'season'], how='left',
    )
    return out

matches_hist = attach_squad_values(matches_hist, squad_vals)
matches_2025 = attach_squad_values(matches_2025, squad_vals)

### 4.5 Caractéristiques statiques des clubs

`clubs_fr.csv` contient des infos sur l'effectif actuel (taille, âge moyen, % d'étrangers, capacité du stade, balance des transferts). Ces variables sont surtout utiles pour les **équipes promues** en 2025-2026 qui ont peu d'historique en Ligue 1.

On parse aussi `net_transfer_record` (chaîne du style `€-72.30m` ou `+€62.11m`) en valeur numérique.

In [ ]:
_NET_TRANSFER_RE = re.compile(r'([+-]?)€?(-?\d+(?:\.\d+)?)([mk]?)', re.IGNORECASE)

def parse_transfer(s) -> float:
    """Convertit '€-72.30m' ou '+€62.11m' en float (euros)."""
    if pd.isna(s):
        return np.nan
    s = str(s).replace(' ', '')
    m = _NET_TRANSFER_RE.search(s)
    if not m:
        return np.nan
    sign, num, unit = m.groups()
    val = float(num)
    if unit.lower() == 'm':
        val *= 1_000_000
    elif unit.lower() == 'k':
        val *= 1_000
    if sign == '-' or s.startswith('€-') or s.startswith('-'):
        val = -abs(val)
    return val

clubs['net_transfer'] = clubs['net_transfer_record'].apply(parse_transfer)
club_cols = ['club_id', 'squad_size', 'average_age', 'foreigners_percentage',
             'national_team_players', 'stadium_seats', 'net_transfer']
c = clubs[club_cols]

for side in ('home', 'away'):
    c_side = c.add_prefix(f'{side}_').rename(columns={f'{side}_club_id': f'{side}_club_id'})
    matches_hist = matches_hist.merge(c_side, on=f'{side}_club_id', how='left')
    matches_2025 = matches_2025.merge(c_side, on=f'{side}_club_id', how='left')

matches_hist.filter(like='squad').head(3)

### 4.6 Jours de repos et features dérivées

Une équipe qui a joué la veille a moins de fraîcheur qu'une équipe qui a une semaine de repos. On calcule le nombre de jours depuis le dernier match de chaque équipe.

On crée aussi quelques **différences** (ELO domicile − ELO extérieur, valeur dom − valeur ext) : ces écarts sont souvent plus parlants que les valeurs brutes pour un modèle linéaire.

In [ ]:
def add_rest_days(matches: pd.DataFrame) -> pd.DataFrame:
    last_seen = {}
    rh, ra = [], []
    for row in matches.sort_values('date').itertuples(index=False):
        d = row.date
        rh.append((d - last_seen[row.home_club_id]).days if row.home_club_id in last_seen else np.nan)
        ra.append((d - last_seen[row.away_club_id]).days if row.away_club_id in last_seen else np.nan)
        last_seen[row.home_club_id] = d
        last_seen[row.away_club_id] = d
    out = matches.sort_values('date').copy()
    out['rest_days_home'] = rh
    out['rest_days_away'] = ra
    return out.sort_index()

matches_hist = add_rest_days(matches_hist)
matches_2025 = add_rest_days(matches_2025)

# Features dérivées : différences entre les deux équipes
for df in (matches_hist, matches_2025):
    df['elo_diff']        = df['home_elo'] - df['away_elo']
    df['squad_value_diff'] = df['home_squad_value'].fillna(0) - df['away_squad_value'].fillna(0)
    df['squad_top3_diff']  = df['home_squad_top3'].fillna(0)  - df['away_squad_top3'].fillna(0)

## 5. Préparation du jeu d'entraînement

On définit la liste finale des features et on fait un **split temporel** :
- **Train** : saisons 2012 à 2022
- **Validation** : saisons 2023 et 2024

On retire les matchs où la forme est encore vide (premiers matchs de chaque équipe sans historique).

**Pourquoi un split temporel et pas aléatoire ?** Parce qu'on veut simuler les conditions réelles : on apprend du passé pour prédire le futur. Un split aléatoire donnerait un score artificiellement bon car le modèle aurait vu des matchs de la même saison dans le train.

In [ ]:
FEATURE_COLUMNS = [
    'elo_diff', 'home_elo', 'away_elo',
    'home_form_pts', 'away_form_pts',
    'home_form_gd', 'away_form_gd',
    'home_form_gf', 'away_form_gf',
    'home_form_ga', 'away_form_ga',
    'h2h_home_winrate', 'h2h_draw_rate', 'h2h_n',
    'squad_value_diff', 'squad_top3_diff',
    'home_squad_value', 'away_squad_value',
    'home_squad_size', 'away_squad_size',
    'home_average_age', 'away_average_age',
    'home_foreigners_percentage', 'away_foreigners_percentage',
    'home_stadium_seats',
    'home_net_transfer', 'away_net_transfer',
    'rest_days_home', 'rest_days_away',
]

df = matches_hist.dropna(subset=['results', 'home_form_pts', 'away_form_pts']).copy()

train = df[df['season'] < 2023]
val   = df[df['season'] >= 2023]

X_train, y_train = train[FEATURE_COLUMNS], train['results'].astype(int)
X_val,   y_val   = val[FEATURE_COLUMNS],   val['results'].astype(int)

print(f'Train : {len(train)} matchs (saisons {train.season.min()}-{train.season.max()})')
print(f'Val   : {len(val)} matchs (saisons {val.season.min()}-{val.season.max()})')

## 6. Modélisation

On compare trois approches :

### Baseline : prédire toujours la classe majoritaire
C'est le score minimum à dépasser pour montrer que le modèle apprend quelque chose.

In [ ]:
base_acc = (y_val == 1).mean()
print(f'[baseline] Toujours "victoire domicile" -> accuracy {base_acc:.3f}')

### Modèle 1 : Régression logistique multinomiale

Modèle **linéaire** : il combine les features par une somme pondérée. Avantages :
- Rapide, interprétable
- Robuste si peu de données
- Bonne baseline

Inconvénients : ne capture pas les interactions complexes entre features.

On **standardise** d'abord les features (moyenne 0, écart-type 1) et on **impute les NaN** par la médiane, car la logistique ne sait pas gérer les valeurs manquantes.

In [ ]:
medians = X_train.median()

logreg = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(max_iter=3000, C=0.3)),  # C=0.3 : régularisation L2 modérée
])
logreg.fit(X_train.fillna(medians), y_train)

proba_lr = logreg.predict_proba(X_val.fillna(medians))
pred_lr  = logreg.predict(X_val.fillna(medians))
acc_lr   = accuracy_score(y_val, pred_lr)
ll_lr    = log_loss(y_val, proba_lr, labels=logreg.classes_)
print(f'[logreg] accuracy={acc_lr:.3f}  log_loss={ll_lr:.3f}')

### Modèle 2 : Gradient Boosting (HistGradientBoosting)

Modèle **non linéaire** basé sur des arbres de décision boostés. Avantages :
- Capture les interactions complexes
- Gère **nativement les NaN** (pas besoin d'imputation)
- Pas de standardisation nécessaire

On le **régularise fortement** (`max_depth=4`, `min_samples_leaf=80`, `l2_regularization=2.0`) pour éviter le surapprentissage, fréquent sur les petits jeux de données. L'**early stopping** arrête l'entraînement quand la validation interne stagne.

In [ ]:
gbm = HistGradientBoostingClassifier(
    max_iter=600,
    learning_rate=0.03,
    max_depth=4,
    min_samples_leaf=80,
    l2_regularization=2.0,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=30,
    random_state=42,
)
gbm.fit(X_train, y_train)

proba_gbm = gbm.predict_proba(X_val)
pred_gbm  = gbm.predict(X_val)
acc_gbm   = accuracy_score(y_val, pred_gbm)
ll_gbm    = log_loss(y_val, proba_gbm, labels=gbm.classes_)
print(f'[hgbm] accuracy={acc_gbm:.3f}  log_loss={ll_gbm:.3f}  (arrêté à {gbm.n_iter_} itérations)')

### Modèle 3 : Ensemble (moyenne des probabilités)

Astuce simple mais souvent efficace : on **moyenne les probabilités** des deux modèles. Les erreurs des deux modèles sont en partie indépendantes, donc l'ensemble est en général plus fiable. C'est très utilisé en compétition (Kaggle, etc.).

In [ ]:
# Les deux modèles ont les mêmes classes dans le même ordre : -1, 0, 1
assert list(logreg.classes_) == list(gbm.classes_)

proba_ens = 0.5 * proba_lr + 0.5 * proba_gbm
pred_ens  = gbm.classes_[proba_ens.argmax(axis=1)]
acc_ens   = accuracy_score(y_val, pred_ens)
ll_ens    = log_loss(y_val, proba_ens, labels=gbm.classes_)
print(f'[ensemble] accuracy={acc_ens:.3f}  log_loss={ll_ens:.3f}')

## 7. Comparaison et choix du modèle final

On sélectionne le modèle avec le **meilleur log-loss** (plus robuste que l'accuracy seule, car il tient compte de la confiance des probabilités prédites).

In [ ]:
results = pd.DataFrame({
    'modèle':   ['baseline', 'logreg', 'hgbm', 'ensemble'],
    'accuracy': [base_acc,   acc_lr,    acc_gbm, acc_ens],
    'log_loss': [np.nan,     ll_lr,     ll_gbm,  ll_ens],
})
print(results.round(3).to_string(index=False))
print()
print('[ensemble] rapport détaillé sur la validation :')
print(classification_report(y_val, pred_ens, digits=3))

**Observation importante** : le modèle prédit très peu de matchs nuls. C'est un problème classique du foot — les nuls sont des évènements "centraux" entre deux camps, donc difficiles à distinguer d'une victoire serrée. Le rappel sur la classe 0 est faible.

## 8. Importance des features

On utilise la **permutation importance** : on mélange aléatoirement chaque feature et on regarde de combien la performance chute. Plus la chute est grande, plus la feature est importante.

In [ ]:
from sklearn.inspection import permutation_importance

r = permutation_importance(gbm, X_val, y_val, n_repeats=5, random_state=42, n_jobs=1)
imp = pd.Series(r.importances_mean, index=FEATURE_COLUMNS).sort_values(ascending=False)
print('Top 10 features les plus importantes :')
print(imp.head(10).round(4))

## 9. Prédiction finale sur la saison 2025-2026

On **ré-entraîne** chaque modèle sur **tout** l'historique disponible (train + validation), puis on combine en ensemble pour prédire les matchs de 2025-2026.

In [ ]:
X_full = df[FEATURE_COLUMNS]
y_full = df['results'].astype(int)
medians_full = X_full.median()

# Ré-entraînement logistique
logreg_final = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(max_iter=3000, C=0.3)),
])
logreg_final.fit(X_full.fillna(medians_full), y_full)

# Ré-entraînement gradient boosting
gbm_final = HistGradientBoostingClassifier(
    max_iter=600, learning_rate=0.03, max_depth=4,
    min_samples_leaf=80, l2_regularization=2.0,
    early_stopping=True, validation_fraction=0.15, n_iter_no_change=30,
    random_state=42,
)
gbm_final.fit(X_full, y_full)

# Prédiction ensemble sur les matchs 2025-2026
X_future = matches_2025[FEATURE_COLUMNS]
proba_lr_fut  = logreg_final.predict_proba(X_future.fillna(medians_full))
proba_gbm_fut = gbm_final.predict_proba(X_future)
proba_fut     = 0.5 * proba_lr_fut + 0.5 * proba_gbm_fut
preds_fut     = gbm_final.classes_[proba_fut.argmax(axis=1)]

predictions = pd.DataFrame({
    'game_id': matches_2025['game_id'].values,
    'results': preds_fut.astype(int),
})
predictions.to_csv(DATA_DIR / 'predictions.csv', index=False)

# On sauvegarde aussi les probabilités (utile pour l'analyse)
proba_df = pd.DataFrame(proba_fut, columns=[f'p_{c}' for c in gbm_final.classes_])
proba_df.insert(0, 'game_id', matches_2025['game_id'].values)
proba_df.to_csv(DATA_DIR / 'predictions_proba.csv', index=False)

print(f'Prédictions écrites dans predictions.csv ({len(predictions)} lignes)')
print(f'Distribution des prédictions :')
print(predictions['results'].value_counts(normalize=True).round(3))

## 10. Aperçu des prédictions

Quelques matchs avec les probabilités de chaque résultat.

In [ ]:
preview = matches_2025[['date', 'home_club_name', 'away_club_name']].copy()
preview['prediction'] = preds_fut
preview['p_dom (1)'] = proba_fut[:, list(gbm_final.classes_).index(1)].round(2)
preview['p_nul (0)'] = proba_fut[:, list(gbm_final.classes_).index(0)].round(2)
preview['p_ext (-1)'] = proba_fut[:, list(gbm_final.classes_).index(-1)].round(2)
preview.head(15)

## 11. Conclusion

### Résultats
Sur la validation 2023-2024, l'ensemble logreg + gradient boosting atteint **~52.5 % d'accuracy**, contre 43 % pour la baseline triviale. Le log-loss est de ~1.00.

### Features les plus prédictives
1. **Différence de valeur des 3 meilleurs joueurs** entre les deux équipes
2. **Différence de valeur totale du squad**
3. **Différence d'ELO**
4. Bilan transferts de l'équipe extérieure
5. Buts encaissés récents par l'équipe à domicile

→ Ce sont avant tout des **différences de niveau global** qui pilotent les prédictions, plutôt que la forme à court terme.

### Limites
- **Prédiction des nuls très faible** (problème intrinsèque du foot)
- **Équipes promues** en 2025-2026 : peu d'historique récent en Ligue 1, donc ELO et forme moins fiables. Les caractéristiques statiques de `clubs_fr.csv` compensent partiellement.
- Pas d'information sur les **compositions** (blessures, suspensions) pour 2025 — c'était une contrainte du sujet.

### Pistes d'amélioration
- Approximer un **xG** (expected goals) en agrégeant les évènements de `game_events_before2025.csv`
- Modèle séparé par équipe (un classifieur par club) si on a assez de données
- Calibration des probabilités (Platt scaling, isotonique) pour mieux refléter la distribution réelle des résultats

### Livrable
Le fichier **`predictions.csv`** au format `game_id,results` (1 / 0 / −1) est le rendu attendu pour le projet.